# Phase 3 - Complaint Classification

Goal: train a TF-IDF + Logistic Regression classifier on the top-20 complaint categories. Target macro-F1 >= 0.75. Save a portable artifact that the deployed Streamlit app can load without Spark.

Phases 0-2 must be passing. Phase 2's `sample_2m_preprocessed.parquet` must be on Drive.

## Cell 1 - Bootstrap (drive + repo + spark + nltk)

In [ ]:
REPO_URL = 'https://github.com/george-gideon-S/cs-gy-6513-big-data-311-nlp.git'

from google.colab import drive
drive.mount('/content/drive')

import subprocess, os, sys
if not os.path.isdir('/content/project/.git'):
    subprocess.run(['git', 'clone', REPO_URL, '/content/project'], check=True)
else:
    subprocess.run(['git', '-C', '/content/project', 'pull'], check=True)

if '/content/project' not in sys.path:
    sys.path.insert(0, '/content/project')

!pip install -r /content/project/requirements-train.txt -q

!apt-get install -y openjdk-11-jre-headless > /dev/null 2>&1
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'
os.environ['PATH'] = os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']

# nltk to /root/nltk_data so workers find it via default search path
import nltk
for pkg in ['stopwords', 'wordnet', 'punkt', 'punkt_tab', 'omw-1.4']:
    nltk.download(pkg, download_dir='/root/nltk_data', quiet=True)

from src.spark_setup import get_spark
spark = get_spark(app_name='phase3-classify')
print('spark', spark.version, 'ready')

## Cell 2 - Load preprocessed data, filter to top-20 classes

We drop empty-token rows (3.2% of corpus) and keep only the top 20 canonical categories. The remaining ~220 small categories together form less than half the corpus and would dilute training signal on the categories we actually evaluate.

In [ ]:
from pyspark.sql import functions as F
from src.config import TOP_K_CATEGORIES

in_path = '/content/drive/MyDrive/cs6513/sample_2m_preprocessed.parquet'
df_full = spark.read.parquet(in_path)
print(f'loaded {df_full.count():,} rows from preprocessed parquet')

# filter empty token lists - they cant be classified anyway
df_clean = df_full.filter(F.size('tokens') > 0)
print(f'after empty-token filter: {df_clean.count():,} rows')

# pick top-20 canonical categories by count
top_classes = (
    df_clean.groupBy('label_canonical')
    .count()
    .orderBy(F.desc('count'))
    .limit(TOP_K_CATEGORIES)
    .toPandas()
)
top_class_list = top_classes['label_canonical'].tolist()
print(f'\ntop {TOP_K_CATEGORIES} classes ({sum(top_classes["count"]):,} rows total):')
print(top_classes.to_string(index=False))

# filter to those classes
df = df_clean.filter(F.col('label_canonical').isin(top_class_list))
n_train_pool = df.count()
print(f'\nfinal training pool: {n_train_pool:,} rows')

## Cell 3 - Stratified 80/20 split

We use sampleBy to maintain class proportions. randomSplit doesnt stratify so smaller classes can vanish from one split.

In [ ]:
from src.classify import stratified_split

train, test = stratified_split(df, label_col='label_canonical', test_fraction=0.2)
n_train = train.count()
n_test = test.count()
print(f'train: {n_train:,} rows ({100*n_train/n_train_pool:.1f}%)')
print(f'test:  {n_test:,} rows ({100*n_test/n_train_pool:.1f}%)')

# sanity check - both splits should have all 20 classes
print(f'\ndistinct classes in train: {train.select("label_canonical").distinct().count()}')
print(f'distinct classes in test:  {test.select("label_canonical").distinct().count()}')

## Cell 4 - Train Logistic Regression pipeline

TF-IDF on tokens (16k feature hash space, min_doc_freq=10) + multinomial Logistic Regression. The pipeline serializes cleanly so Phase 7 dashboard can load it via PipelineModel.load. Training time on 1.5M rows: ~3-5 min.

In [ ]:
from src.classify import build_pipeline
import time

pipeline = build_pipeline(label_col='label_canonical', num_features=16384, min_doc_freq=10)

t0 = time.time()
lr_model = pipeline.fit(train)
t_fit = time.time() - t0
print(f'logistic regression fit in {t_fit:.1f} sec')

## Cell 5 - Evaluate on test set

Macro-F1 is the headline metric (proposal target: 0.75). Accuracy and weighted-F1 are reported alongside for completeness.

In [ ]:
from src.classify import evaluate

lr_metrics = evaluate(lr_model, test)
print('logistic regression test metrics:')
for k, v in lr_metrics.items():
    print(f'  {k:25s} = {v:.4f}')

# headline check vs proposal target
print(f'\nproposal target: macro-F1 >= 0.75')
print(f'we got:          macro-F1 = {lr_metrics["f1"]:.4f}')
if lr_metrics['f1'] >= 0.75:
    print('PASSED')
else:
    print('below target - random forest comparison + tuning may close the gap')

## Cell 6 - Per-class metrics + confusion matrix

Per-class precision/recall surface where the model struggles. The confusion matrix saved to `dashboard/assets/cm.png` becomes the Bias Audit tab's visual.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

preds = lr_model.transform(test).select('label_canonical', 'prediction', 'label').toPandas()

# label index <-> name mapping from the StringIndexerModel
indexer_labels = lr_model.stages[0].labels
preds['pred_name'] = preds['prediction'].astype(int).map(lambda i: indexer_labels[i])

# per-class precision / recall / f1
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
p, r, f1, support = precision_recall_fscore_support(
    preds['label_canonical'], preds['pred_name'], labels=indexer_labels, zero_division=0
)
per_class = pd.DataFrame({
    'class': indexer_labels,
    'precision': p.round(3),
    'recall': r.round(3),
    'f1': f1.round(3),
    'support': support,
}).sort_values('support', ascending=False)
print('per-class metrics:')
print(per_class.to_string(index=False))

# save for the dashboard
per_class.to_json('/content/project/dashboard/assets/per_class_metrics.json', orient='records', indent=2)
print('\nsaved per_class_metrics.json')

# confusion matrix
cm = confusion_matrix(preds['label_canonical'], preds['pred_name'], labels=indexer_labels)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(cm_norm, cmap='Purples', aspect='auto')
ax.set_xticks(range(len(indexer_labels)))
ax.set_yticks(range(len(indexer_labels)))
ax.set_xticklabels(indexer_labels, rotation=60, ha='right', fontsize=8)
ax.set_yticklabels(indexer_labels, fontsize=8)
ax.set_xlabel('predicted')
ax.set_ylabel('actual')
ax.set_title(f'normalized confusion matrix - macro-F1 = {lr_metrics["f1"]:.3f}')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig('/content/project/dashboard/assets/cm.png', dpi=120, bbox_inches='tight')
plt.show()
print('saved cm.png')

## Cell 7 - Baselines (majority class + keyword heuristic)

Required by the proposal. Shows our trained model actually adds value over trivial heuristics.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

y_true = preds['label_canonical'].values

# baseline 1: always predict the majority class
majority = top_classes['label_canonical'].iloc[0]
y_pred_majority = np.full_like(y_true, majority, dtype=object)
f1_maj = f1_score(y_true, y_pred_majority, average='macro', zero_division=0)
acc_maj = accuracy_score(y_true, y_pred_majority)
print(f'majority-class baseline:    macro-F1 = {f1_maj:.4f}, accuracy = {acc_maj:.4f}')

# baseline 2: keyword match against the top 3 most distinctive terms per class.
# build a tiny keyword map on the fly from the training split.
from collections import Counter
keyword_map = {}
for cls in indexer_labels:
    cls_tokens = (
        train.filter(F.col('label_canonical') == cls)
        .select(F.explode('tokens').alias('t'))
        .groupBy('t').count()
        .orderBy(F.desc('count'))
        .limit(3)
        .toPandas()['t'].tolist()
    )
    keyword_map[cls] = set(cls_tokens)

# score each test row against each class by token overlap, predict highest
test_pdf = test.select('label_canonical', 'tokens').toPandas()
def keyword_predict(toks):
    toks_set = set(toks)
    best_cls = None
    best_score = -1
    for cls, kws in keyword_map.items():
        score = len(toks_set & kws)
        if score > best_score:
            best_score = score
            best_cls = cls
    return best_cls

test_pdf['kw_pred'] = test_pdf['tokens'].apply(keyword_predict)
f1_kw = f1_score(test_pdf['label_canonical'], test_pdf['kw_pred'], average='macro', zero_division=0)
acc_kw = accuracy_score(test_pdf['label_canonical'], test_pdf['kw_pred'])
print(f'keyword-heuristic baseline: macro-F1 = {f1_kw:.4f}, accuracy = {acc_kw:.4f}')

print(f'\nlogistic regression:        macro-F1 = {lr_metrics["f1"]:.4f}, accuracy = {lr_metrics["accuracy"]:.4f}')
print(f'lift over majority:         +{lr_metrics["f1"] - f1_maj:.4f} F1')
print(f'lift over keyword:          +{lr_metrics["f1"] - f1_kw:.4f} F1')

## Cell 8 - Save full model + portable export

Two artifacts:
1. **Full PipelineModel on Drive** at `/content/drive/MyDrive/cs6513/models/classifier_lr/`. Phase 8 streaming uses this directly.
2. **Portable .npz at `/content/project/models/portable/classifier.npz`** — pure numpy. The deployed Streamlit app loads this with no Spark dependency.

In [ ]:
from src.classify import export_portable

# full model on drive
model_path = '/content/drive/MyDrive/cs6513/models/classifier_lr'
lr_model.write().overwrite().save(model_path)
print(f'full PipelineModel saved to {model_path}')

# portable artifact in repo
portable_path = '/content/project/models/portable/classifier.npz'
export_portable(lr_model, portable_path)

# check the size - we want this under a few MB so it commits cleanly
import os
size_mb = os.path.getsize(portable_path) / 1024 / 1024
print(f'portable artifact size: {size_mb:.2f} MB')

## Cell 9 - Save metrics summary for the dashboard

The Pipeline Status tab reads this JSON to show "Phase 3 PASSED" with concrete numbers.

In [ ]:
import json, datetime

summary = {
    'phase': 3,
    'trained_at': datetime.datetime.utcnow().isoformat() + 'Z',
    'n_train': int(n_train),
    'n_test': int(n_test),
    'n_classes': len(indexer_labels),
    'classes': list(indexer_labels),
    'model': 'tf-idf + logistic regression (multinomial)',
    'feature_dim': 16384,
    'metrics': {k: float(v) for k, v in lr_metrics.items()},
    'baselines': {
        'majority_class': {'macro_f1': float(f1_maj), 'accuracy': float(acc_maj)},
        'keyword_heuristic': {'macro_f1': float(f1_kw), 'accuracy': float(acc_kw)},
    },
    'lift_over_majority_f1': float(lr_metrics['f1'] - f1_maj),
    'lift_over_keyword_f1': float(lr_metrics['f1'] - f1_kw),
    'training_time_sec': float(t_fit),
}
with open('/content/project/dashboard/assets/classifier_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print('saved classifier_summary.json:')
print(json.dumps(summary, indent=2, default=str))

## Cell 10 - Push artifacts to GitHub

Commits the classifier portable .npz + dashboard JSONs + confusion matrix png to the repo. Requires `GITHUB_PAT` in Colab Secrets.

In [ ]:
from src.colab_git import commit_artifacts
commit_artifacts(message='phase 3: classifier portable + metrics')

## Phase 3 - Done when

- Cell 5 prints macro-F1 >= 0.75 (proposal target).
- Cell 7 shows positive lift over both baselines.
- Cell 8 reports portable artifact size under 10 MB (so we can commit it to git).
- `dashboard/assets/cm.png`, `per_class_metrics.json`, `classifier_summary.json` all written.

Save the notebook back to GitHub, drop the new PRINT pdf in the project directory, and we move to Phase 4 (resolution-time regression).